In [1]:
from src.misc import *
from src.SPECTRUM import Spectrum
from src.FITSPECTRUM import FitSpectrum
from src.DP import DP
import matplotlib.pyplot as plt
import numpy as np

In [2]:
spectra_data    = fits.open('/Users/hyp0515/data/0715_Spring_BGS_ALL_trimmed.fits')
cigale_data     = fits.open('/Users/hyp0515/data/IronPhysProp_v1.2_extracted.fits')
fastspecfit     = fits.open('/Users/hyp0515/data/0715_Spring_half_BGS_BRIGHT_catalog_fastspecfit.fits')

# Select parent sources who have at least one significant emission line (>5 S/N ratio)

In [3]:
FIT = FitSpectrum()

ALL_SPECTRA = Spectrum(spectra_data, cigale_data, fastspecfit, load_targetID=read_ids('filtered_ids.txt'))
ALL_SPECTRA = ALL_SPECTRA.subtype_filter(subtype='QSO', exclude=True)
ALL_SPECTRA = ALL_SPECTRA.stack_data()
ALL_SPECTRA = ALL_SPECTRA.shift_to_rest_frame()
print('Total number of spectra:', len(ALL_SPECTRA.targetID))

Total number of spectra: 5860


In [4]:
ALL_SPECTRA = FIT.label_emission_lines(ALL_SPECTRA, 5)
ALL_SPECTRA = FIT.significant_emission_filter(ALL_SPECTRA)
print('Total number of spectra:', len(ALL_SPECTRA.targetID))

Total number of spectra: 5860


In [5]:
# ALL_SPECTRA = ALL_SPECTRA.shrink_dataset(10)
# print('Number of spectra after shrinking:', len(ALL_SPECTRA.targetID))

In [6]:
DP = DP()
dp_parent, model_1comp, left_2comp, right_2comp = DP.fit_all(data_class=ALL_SPECTRA, n_jobs=10)
DP.get_catalog(df=dp_parent, fname='all_catalog.fits', model_1comp=model_1comp, left_2comp=left_2comp, right_2comp=right_2comp)


dp_sample, model_1comp, left_2comp, right_2comp = DP.select_dp_sample(dp_parent, model_1comp, left_2comp, right_2comp)
DP.get_catalog(df=dp_sample, fname='dp_catalog.fits', model_1comp=model_1comp, left_2comp=left_2comp, right_2comp=right_2comp)


100%|██████████| 5860/5860 [00:47<00:00, 123.40it/s]


In [7]:
test_fits = fits.open('dp_catalog.fits')
test_fits.info()

Filename: dp_catalog.fits
No.    Name      Ver    Type      Cards   Dimensions   Format
  0  PRIMARY       1 PrimaryHDU       4   ()      
  1  DATA          1 BinTableHDU     75   1793R x 33C   [K, E, E, E, E, E, E, E, E, E, E, E, L, L, L, L, L, L, L, L, L, L, K, K, K, K, K, K, K, K, K, K, K]   
  2  1COMP         1 ImageHDU         8   (7781, 1793)   float32   
  3  2COMP_L       1 ImageHDU         8   (7781, 1793)   float32   
  4  2COMP_R       1 ImageHDU         8   (7781, 1793)   float32   


In [8]:
print(test_fits[2].data.shape)

(1793, 7781)
